# Lecture 10: Forecasting Techniques
In this notebook we will cover:
- ARAR Algorithm (memory shortening + subset AR)
- Simple Exponential Smoothing (SES)
- Holt's Linear Trend Method
- Holt-Winters Seasonal Method (Additive + Multiplicative)
- Method comparison (benchmark)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Ready.")


## 10.1 ARAR Algorithm

**ARAR** = **AR**ma + **AR** (Brockwell & Davis, Chapter 10.1)

Two-step approach:
1. **Memory shortening:** Remove long memory via the regression $X_t \approx \hat{\phi} X_{t-\tau}$
2. **AR fitting:** Fit a subset AR model to the shortened series

**Step 1 — Find the dominant lag:**
$$\hat{\phi}(\tau) = \frac{\sum_{t=\tau+1}^n X_t X_{t-\tau}}{\sum_{t=\tau+1}^n X_{t-\tau}^2}$$


In [ ]:
# ── ARAR Algorithm: Manual Implementation ──

def arar_prewhiten(X, max_tau=15):
    # Step 1: Find the dominant lag
    n = len(X)
    acf_vals = acf(X, nlags=max_tau, fft=True)
    abs_acf = np.abs(acf_vals[1:])
    tau_star = np.argmax(abs_acf) + 1
    
    # AR(1)-like filter: X_t = phi * X_{t-tau} + residual
    X_lag = X[:-tau_star]
    X_cur = X[tau_star:]
    phi_hat = np.dot(X_cur, X_lag) / np.dot(X_lag, X_lag)
    
    # Compute residuals
    residuals = X_cur - phi_hat * X_lag
    
    return residuals, tau_star, phi_hat

def subset_ar_fit(W, max_p=20, n_terms=5):
    # Subset AR: select the most significant lags
    n = len(W)
    acf_vals = acf(W, nlags=max_p, fft=True)
    abs_acf = np.abs(acf_vals[1:])
    
    # Take the top n_terms lags
    top_lags = np.argsort(abs_acf)[-n_terms:] + 1
    top_lags = sorted(top_lags)
    
    # Estimate coefficients via OLS
    max_lag = max(top_lags)
    rows = []
    for t in range(max_lag, n):
        row = [W[t-l] for l in top_lags]
        rows.append(row)
    Z_mat = np.array(rows)
    y_vec = W[max_lag:]
    
    coeffs, _, _, _ = np.linalg.lstsq(Z_mat, y_vec, rcond=None)
    fitted = Z_mat @ coeffs
    sigma2 = np.var(y_vec - fitted)
    
    return coeffs, top_lags, sigma2, max_lag

def arar_forecast(X, h=20, max_tau=15, n_ar_terms=5):
    W, tau, phi = arar_prewhiten(X, max_tau)
    coeffs, lags, sigma2, max_lag = subset_ar_fit(W, n_terms=n_ar_terms)
    
    # Iterative forecasts
    W_ext = list(W.copy())
    forecasts_W = []
    for _ in range(h):
        max_l = max(lags)
        row = [W_ext[-l] for l in lags]
        pred = np.dot(coeffs, row)
        forecasts_W.append(pred)
        W_ext.append(pred)
    
    return np.array(forecasts_W), W, tau, phi, sigma2

print("ARAR functions ready.")


In [ ]:
# ── ARAR Forecast on AirPassengers ──

# AirPassengers data
ap = np.array([112,118,132,129,121,135,148,148,136,119,104,118,
               115,126,141,135,125,149,170,170,158,133,114,140,
               145,150,178,163,172,178,199,199,184,162,146,166,
               171,180,193,181,183,218,230,242,209,191,172,194,
               196,196,236,235,229,243,264,272,237,211,180,201,
               204,188,235,227,234,264,302,293,259,229,203,229,
               242,233,267,269,270,315,364,347,312,274,237,278,
               284,277,317,313,318,374,413,405,355,306,271,306,
               315,301,356,348,355,422,465,467,404,347,305,336,
               340,318,362,348,363,435,491,505,404,359,310,337,
               360,342,406,396,420,472,548,559,463,407,362,405,
               417,391,419,461,472,535,622,606,508,461,390,432])

log_ap = np.log(ap)
n_train = len(log_ap) - 24
train = log_ap[:n_train]
test  = log_ap[n_train:]

preds_arar, W_arar, tau_star, phi_hat, sigma2_arar = arar_forecast(train, h=24)

# Note: this is a simplified ARAR implementation — full back-transformation
# requires unbiased AR iteration; here we compare W forecasts directly
# with log-AP for demonstration purposes.

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(log_ap, color='steelblue', lw=1.5, label='Log AP (Actual)', zorder=3)
ax.axvline(n_train, color='black', ls='--', alpha=0.5, label='Train/Test boundary')
ax.set_title(f'ARAR: Memory shortening tau={tau_star}, phi={phi_hat:.3f}', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Log Passenger Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Dominant lag    : tau* = {tau_star}")
print(f"Memory shortening coefficient: phi = {phi_hat:.4f}")


## 10.2 Exponential Smoothing Methods

### Simple Exponential Smoothing (SES)
$$\hat{X}_{t+1} = \alpha X_t + (1-\alpha)\hat{X}_t, \quad 0 < \alpha < 1$$

- $\alpha \to 1$: only the most recent observation is used
- $\alpha \to 0$: all past observations receive equal weight

### Holt's Method (Linear Trend)
$$\hat{X}_{t+h} = l_t + h \cdot b_t$$
$$l_t = \alpha X_t + (1-\alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1-\beta)b_{t-1}$$

### Holt-Winters (Seasonal)
**Additive:** $\hat{X}_{t+h} = l_t + h b_t + s_{t+h-m}$

**Multiplicative:** $\hat{X}_{t+h} = (l_t + h b_t) \cdot s_{t+h-m}$


In [ ]:
# ── SES, Holt, Holt-Winters Comparison ──

import pandas as pd
t_idx = pd.date_range('1949-01', periods=len(ap), freq='MS')
ap_series = pd.Series(ap, index=t_idx)

train_s = ap_series.iloc[:n_train]
test_s  = ap_series.iloc[n_train:]

# 1. SES
ses = ExponentialSmoothing(train_s, trend=None, seasonal=None).fit(optimized=True)
# 2. Holt
holt = ExponentialSmoothing(train_s, trend='add', seasonal=None).fit(optimized=True)
# 3. Holt-Winters Additive
hw_add = ExponentialSmoothing(train_s, trend='add', seasonal='add', seasonal_periods=12).fit(optimized=True)
# 4. Holt-Winters Multiplicative
hw_mul = ExponentialSmoothing(train_s, trend='add', seasonal='mul', seasonal_periods=12).fit(optimized=True)

h = len(test_s)
fc_ses  = ses.forecast(h)
fc_holt = holt.forecast(h)
fc_add  = hw_add.forecast(h)
fc_mul  = hw_mul.forecast(h)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
methods = [
    ('SES', fc_ses, 'steelblue'),
    ('Holt (Trend)', fc_holt, 'darkorange'),
    ('HW Additive', fc_add, 'green'),
    ('HW Multiplicative', fc_mul, 'purple'),
]

for ax, (name, fc, color) in zip(axes.flat, methods):
    ax.plot(ap_series, color='gray', lw=0.8, alpha=0.5, label='Actual')
    ax.plot(fc, color=color, lw=2, label=name)
    ax.axvline(train_s.index[-1], color='black', ls='--', alpha=0.5)
    rmse = np.sqrt(np.mean((fc.values - test_s.values)**2))
    ax.set_title(f'{name} | RMSE={rmse:.1f}', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Comprehensive Benchmark ──

from sklearn.metrics import mean_squared_error, mean_absolute_error

def smape(actual, predicted):
    return 100 * np.mean(2*np.abs(predicted-actual) / (np.abs(actual)+np.abs(predicted)))

results = {}
models = {
    'SES': fc_ses,
    'Holt': fc_holt,
    'HW-Add': fc_add,
    'HW-Mul': fc_mul,
}

# Add ARIMA(1,1,1)(1,1,1)_12
from statsmodels.tsa.statespace.sarimax import SARIMAX
sarima = SARIMAX(np.log(train_s), order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False)
fc_sarima = np.exp(sarima.forecast(h))
models['SARIMA'] = fc_sarima

for name, fc in models.items():
    rmse  = np.sqrt(mean_squared_error(test_s, fc))
    mae   = mean_absolute_error(test_s, fc)
    smape_val = smape(test_s.values, fc.values)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'sMAPE(%)': smape_val}

results_df = pd.DataFrame(results).T.sort_values('RMSE')
print("\nMethod Comparison (24-Step Forecast):")
print(results_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if x == results_df['RMSE'].min() else 'lightgray'
          for x in results_df['RMSE']]
bars = ax.barh(results_df.index, results_df['RMSE'], color=colors)
ax.set_title('RMSE Comparison (Lower = Better)', fontweight='bold')
ax.set_xlabel('RMSE')
for bar, val in zip(bars, results_df['RMSE']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

best = results_df.index[0]
print(f"\nBest method: {best} (RMSE={results_df['RMSE'].iloc[0]:.2f})")


## 10.3 Parameter Optimisation: Effect of Alpha

Let us examine the effect of the SES parameter $\alpha$ on forecast performance.


In [ ]:
# ── Effect of Alpha ──

alphas = [0.05, 0.2, 0.5, 0.8, 0.95]
colors_a = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ap_series, color='black', lw=1.5, alpha=0.5, label='Actual', zorder=10)

for alpha, col in zip(alphas, colors_a):
    ses_a = ExponentialSmoothing(train_s, trend=None, seasonal=None).fit(
        smoothing_level=alpha, optimized=False)
    fc_a = ses_a.forecast(h)
    ax.plot(fc_a, color=col, lw=1.5, ls='--', alpha=0.8, label=f'alpha={alpha}')

ax.axvline(train_s.index[-1], color='gray', ls='--')
ax.set_title('SES: 24-Step Forecast with Different alpha Values', fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
plt.tight_layout()
plt.show()

print("Small alpha: slow response, smoother forecast")
print("Large alpha: fast response, more noise")


## Summary: Choosing a Forecasting Method

| Method | When to Use | Parameters |
|--------|-------------|------------|
| **SES** | No trend, no seasonality | $\alpha$ |
| **Holt** | Trend present, no seasonality | $\alpha, \beta$ |
| **HW-Add** | Constant-amplitude seasonality | $\alpha, \beta, \gamma$ |
| **HW-Mul** | Growing-amplitude seasonality | $\alpha, \beta, \gamma$ |
| **SARIMA** | Strong seasonal structure, large dataset | $p,d,q,P,D,Q,s$ |
| **ARAR** | Long memory, complex autocorrelation | $\tau, \phi$ |

**Practical Guide:**
1. Visualise the data → trend? seasonality?
2. If seasonality is present → HW-Mul or SARIMA
3. Run a comparative benchmark (RMSE / MAE / sMAPE)
4. Choose based on forecast horizon and data size
